In [1]:
%reload_ext autoreload
%autoreload 2
import json
from pathlib import Path
from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

from pyeed.core import ProteinRecord

In [4]:
from nxontology.imports import from_file
import networkx as nx
from collections import Counter
import pronto
from nxontology import NXOntology
from nxontology.imports import pronto_to_multidigraph, multidigraph_to_digraph

# Step 1: Parse the Gene Ontology
url = "http://release.geneontology.org/2021-02-01/ontology/go-basic.json.gz"
nxo = from_file(url)
print(nxo.n_nodes)  # Output the number of nodes

# Calculate similarity between two GO terms
sim = nxo.similarity("GO:0042552", "GO:0022008")
print(round(sim.lin, 2))  # Output: 0.21

# Check the number of weakly connected components
print(nx.number_weakly_connected_components(nxo.graph))  # Output: 3

# Preserve all GO relationship types
go_pronto = pronto.Ontology(handle=url)
go_multidigraph = pronto_to_multidigraph(go_pronto)
print(Counter(key for _, _, key in go_multidigraph.edges(keys=True)))
# Output: Counter({'is a': 71509, 'part of': 7187, 'regulates': 3216,
# 'negatively regulates': 2768, 'positively regulates': 2756})

go_digraph = multidigraph_to_digraph(go_multidigraph, reduce=True)
go_nxo = NXOntology(go_digraph)

# Notice the similarity increases due to the full set of edges
print(round(go_nxo.similarity("GO:0042552", "GO:0022008").lin, 3))  # Output: 0.699

# Dedicated reader for the Gene Ontology
from nxontology.imports import read_gene_ontology

go_nxo_full = read_gene_ontology(release="2021-02-01")

# Step 2: Convert the NXOntology graph to a NetworkX graph
G = go_nxo.graph

# Step 3: Perform operations or visualizations on the NetworkX graph

# Example: Print the attributes of a specific node
node_id = "GO:0000128"
if node_id in G.nodes:
    node_attributes = G.nodes[node_id]
    print(f"Attributes of node {node_id}: {node_attributes}")
else:
    print(f"Node {node_id} does not exist in the graph.")

44085
0.21
3
Counter({'is a': 71509, 'part of': 7187, 'regulates': 3216, 'negatively regulates': 2768, 'positively regulates': 2756})
0.699
Attributes of node GO:0000128: {'identifier': 'GO:0000128', 'name': 'flocculation', 'namespace': 'biological_process'}


In [1]:
from nxontology.imports import from_file

# versioned URL for the Gene Ontology
url = "http://release.geneontology.org/2021-02-01/ontology/go-basic.json.gz"
nxo = from_file(url)
print(nxo.n_nodes)  # Output: 44085

# similarity between "myelination" and "neurogenesis"
sim = nxo.similarity("GO:0042552", "GO:0022008")
print(round(sim.lin, 2))  # Output: 0.21

import networkx as nx

# Gene Ontology domains are disconnected, expect 3 components
print(nx.number_weakly_connected_components(nxo.graph))  # Output: 3

# Note however that the default from_file reader only uses "is a" relationships.
# We can preserve all GO relationship types as follows
from collections import Counter
import pronto
from nxontology import NXOntology
from nxontology.imports import pronto_to_multidigraph, multidigraph_to_digraph

go_pronto = pronto.Ontology(handle=url)
go_multidigraph = pronto_to_multidigraph(go_pronto)
print(Counter(key for _, _, key in go_multidigraph.edges(keys=True)))
# Output: Counter({'is a': 71509, 'part of': 7187, 'regulates': 3216,
# 'negatively regulates': 2768, 'positively regulates': 2756})

go_digraph = multidigraph_to_digraph(go_multidigraph, reduce=True)
go_nxo = NXOntology(go_digraph)

# Notice the similarity increases due to the full set of edges
print(round(go_nxo.similarity("GO:0042552", "GO:0022008").lin, 3))  # Output: 0.699

# Note that there is also a dedicated reader for the Gene Ontology
from nxontology.imports import read_gene_ontology

go_nxo_full = read_gene_ontology(release="2021-02-01")

44085
0.21
3
Counter({'is a': 71509, 'part of': 7187, 'regulates': 3216, 'negatively regulates': 2768, 'positively regulates': 2756})
0.699


In [2]:
# get pyeed logger to log to file
import logging

logging.getLogger("pyeed").setLevel(logging.DEBUG)
# logger to log to file
logger = logging.getLogger("pyeed")
logger.setLevel(logging.DEBUG)
# create file handler which logs even debug messages
fh = logging.FileHandler("pyeed.log")
fh.setLevel(logging.ERROR)
# connect logger with file handler
logger.addHandler(fh)

## Get `ProteinRecord` instances

In [3]:
with open("../../docs/examples/ids.json") as f:
    ids = json.load(f)

seqs = ProteinRecord.get_ids(ids)

Output()

In [12]:
print(seqs[0])

ProteinRecord
├── id = Q58605
├── name = S-adenosylmethionine synthase
├── organism
│   └── Organism
│       ├── id = 1edef44d-d991-465c-b753-ef5815d2e6a8
│       ├── taxonomy_id = 243232
│       ├── name = Methanocaldococcus jannaschii DSM 2661
│       ├── domain = Archaea
│       ├── phylum = Euryarchaeota
│       ├── tax_class = Methanococci
│       ├── order = Methanococcales
│       ├── family = Methanocaldococcaceae
│       └── genus = Methanocaldococcus
├── sequence = MRNIIVKKLDVEPIEERPTEIVERKGLGHPDSICDGIAESVSRALCKMYMEKFGTILHHNTDQVELVGGHAYPKFGGGVMVSPIYILLSGRATMEILDKEKNEVIKLPVGTTAVKAAKEYLKKVLRNVDVDKDVIIDCRIGQGSMDLVDVFERQKNEVPLANDTSFGVGYAPLSTTERLVLETERFLNSDELKNEIPAVGEDIKVMGLREGKKITLTIAMAVVDRYVKNIEEYKEVIEKVRKKVEDLAKKIADGYEVEIHINTADDYERESVYLTVTGTSAEMGDDGSVGRGNRVNGLITPFRPMSMEAASGKNPVNHVGKIYNILANLIANDIAKLEGVKECYVRILSQIGKPINEPKALDIEIITEDSYDIKDIEPKAKEIANKWLDNIMEVQKMIVEGKVTTF
├── regions
│   ├── 0
│   │   └── Region
│   │       ├── id = 50ea7783-1625-4b70-b2f9-7968aa2b12d3
│   │       ├─

In [11]:
print(seqs[0].json())

{
  "@context": {
    "ProteinRecord": "https://github.com/PyEED/pyeed/ProteinRecord",
    "coding_sequence": {
      "@container": "@list",
      "@id": "http://semanticscience.org/resource/SIO_001390"
    },
    "ec_number": "http://edamontology.org/data_1011",
    "mol_weight": "http://edamontology.org/data_1505",
    "name": "http://semanticscience.org/resource/SIO_000116",
    "organism": "http://semanticscience.org/resource/SIO_010000",
    "seq_length": "http://semanticscience.org/resource/SIO_000041",
    "sequence": "http://semanticscience.org/resource/SIO_000030",
    "structure_id": "http://semanticscience.org/resource/SIO_000729"
  },
  "@id": "Q58605",
  "@type": [
    "ProteinRecord"
  ],
  "coding_sequence": [
    {
      "@context": {
        "Region": "https://github.com/PyEED/pyeed/Region",
        "accession_id": "http://semanticscience.org/resource/SIO_000675",
        "end": "http://semanticscience.org/resource/SIO_000953",
        "start": "http://semanticscience.

In [4]:
# write json-ld files
for seq in seqs:
    with open(f"../json/{seq.id}.json", "w") as f:
        f.write(seq.json())

# write ttl files
for seq in seqs:
    g = Graph()
    g.parse(data=seq.json(), format="json-ld")
    g.serialize(destination=f"../ttl/{seq.id}.ttl", format="turtle")

## Connect to neo4j database

In [5]:
# set the configuration to connect to your Aura DB
AURA_DB_URI = "bolt://localhost:7687"

auth_data = {
    "uri": AURA_DB_URI,
    "database": "neo4j",
    "user": "None",
    "pwd": "None",
}

# Define your custom mappings & store config
config = Neo4jStoreConfig(
    auth_data=auth_data,
    # custom_prefixes=prefixes,
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=True,
)

# Create the RDF Graph, parse & ingest the data to Neo4j, and close the store
# (If the field batching is set to True in the Neo4jStoreConfig, remember to close the store to prevent
# the loss of any uncommitted records.)
neo4j_aura = Graph(store=Neo4jStore(config=config))
# Calling the parse method will implictly open the store


for path in Path("../ttl").rglob("*.ttl"):
    neo4j_aura.parse(str(path), format="ttl")

neo4j_aura.close(True)

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ()):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 61] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 61] Connection refused)